In [12]:
using DriftDiffusionModels
using HiddenMarkovModels
using LinearAlgebra
using CSV
using DataFrames
using Distributions
using Dates
using Random
using Plots

In [6]:
ddir = "../data/mouse_df.csv"
# load in the data
df = CSV.File(ddir) |> DataFrame;

In [10]:
# Group by animal name and count trials
trial_counts = combine(groupby(df, :name), nrow => :trial_count)

# Sort by count in descending order
sort!(trial_counts, :trial_count, rev=true)

# pick a mouse of interest (start with mouse of most trials)
moi = trial_counts[1, :name]

# get the data for the mouse of interest
mouse_df = df[df.name .== moi, :]

# Filter out trials with "omission" outcome
valid_trials = findall(outcome -> outcome != "omission", mouse_df.outcome)
filtered_df = mouse_df[valid_trials, :]

# Map correct -> 1 and incorrect -> -1 (only for error and correct, omissions are gone)
numeric_outcomes = [outcome == "correct" ? 1 : -1 for outcome in filtered_df.outcome]

# Get reaction times, filter out "NAN" values
valid_rt_indices = findall(rt -> uppercase(string(rt)) != "NAN", filtered_df.rt)
# valid_trials = findall(rt -> rt > 0.2, filtered_df.rt)

# Apply both filters to keep data aligned
final_df = filtered_df[valid_rt_indices, :]
final_outcomes = numeric_outcomes[valid_rt_indices]

# Convert RTs to Float64
# final_rts = [parse(Float64, rt) for rt in final_df.rt]
final_rts = final_df.rt
final_stimulus = final_df.correct_side
final_stimulus = [stim == "right" ? 1 : -1 for stim in final_stimulus]

# Extract just the date part from the timestamp strings
dates = [Date(split(dt)[1]) for dt in final_df.trial_datetime]

# Get unique dates in chronological order
unique_dates = sort(unique(dates))

# Create a vector of vectors, where each inner vector contains DDMResults for one day
results_by_date = Vector{Vector{DDMResult}}()

for date in unique_dates
    # Get indices for this date
    day_indices = findall(dates .== date)
    
    # Skip days with no valid data
    if isempty(day_indices)
        continue
    end
    
    # Extract RTs and outcomes for this date
    day_rts = final_rts[day_indices]
    day_outcomes = final_outcomes[day_indices]
    day_stim_side = final_stimulus[day_indices]
    
    # Create DDMResult objects for this day
    day_results = [DDMResult(rt, choice, stim) for (rt, choice, stim) in zip(day_rts, day_outcomes, day_stim_side)]

    # Add to our vector of vectors
    push!(results_by_date, day_results)
end

# Now calculate the sequence ends (cumulative sum of lengths)
seq_ends = cumsum([length(seq) for seq in results_by_date])

# Concatenate all results into a single vector
all_results = reduce(vcat, results_by_date)



12311-element Vector{DDMResult}:
 DDMResult(0.2716, 1, -1)
 DDMResult(0.4124, -1, 1)
 DDMResult(0.2669, 1, -1)
 DDMResult(0.8428, 1, 1)
 DDMResult(0.3541, 1, 1)
 DDMResult(0.247, 1, -1)
 DDMResult(0.4099, 1, -1)
 DDMResult(0.2372, -1, -1)
 DDMResult(0.3275, -1, 1)
 DDMResult(0.2344, 1, 1)
 ⋮
 DDMResult(0.4864, 1, 1)
 DDMResult(0.2847, -1, -1)
 DDMResult(0.2424, 1, 1)
 DDMResult(0.4899, -1, -1)
 DDMResult(0.226, -1, -1)
 DDMResult(0.2562, -1, -1)
 DDMResult(0.5747, 1, -1)
 DDMResult(0.3208, 1, 1)
 DDMResult(0.4196, 1, -1)

In [36]:
# assume a 3 state model
num_states = 3

α = 10.0 # concentration paramerter for a Dirichlet prior, where α is the main diagonal concentration
A = zeros(num_states, num_states)

for i in 1:num_states
    dir = zeros(num_states)
    for j in 1:num_states
        if i == j
            dir[j] = α
        else
            dir[j] = 1.0
        end
    end
    A[i, :] = rand(Dirichlet(dir))
end

π₀ = rand(Dirichlet(fill(1.0, num_states)))

# Define priors over each paramerter
logv₀_prior = Normal(log(1.0), 0.5)
a₀_prior = Beta(10.0, 10.0)
logB_prior = Normal(log(2.0), 0.5)
τ_init = 0.1

ddms = Vector{DriftDiffusionModel}(undef, num_states)
for i in 1:num_states
    logv₀ = rand(logv₀_prior)
    a₀ = rand(a₀_prior)
    logB = rand(logB_prior)
    ddms[i] = DriftDiffusionModel(exp(logB), exp(logv₀), a₀, τ_init)
end 

In [37]:
αT = ones(num_states, num_states);      αT[diagind(αT)] .= 5   # sticky prior
απ = fill(2.0, num_states)

hmm_init = PriorHMM(π₀, A, ddms, αT, απ)

PriorHMM{Float64, DriftDiffusionModel}([0.3257241115063952, 0.3012576610374629, 0.3730182274561421], [0.8321914675678777 0.04012030793881902 0.1276882244933034; 0.014856270602714397 0.9592162160506075 0.025927513346678; 0.0871225688860687 0.015641086080145014 0.8972363450337862], DriftDiffusionModel[DriftDiffusionModel(0.7176881956929024, 1.6472963487141845, 0.5087202645646632, 0.1), DriftDiffusionModel(2.833180447019718, 1.1778940171787602, 0.3404358638704114, 0.1), DriftDiffusionModel(1.3657786675938937, 1.1932562369091066, 0.5172082165905585, 0.1)], [5.0 1.0 1.0; 1.0 5.0 1.0; 1.0 1.0 5.0], [2.0, 2.0, 2.0])

In [38]:
hmm_est, lls = baum_welch(hmm_init, all_results; seq_ends=seq_ends)

(PriorHMM{Float64, DriftDiffusionModel}([0.1370329881079627, 0.6276356275436396, 0.2353313843483977], [0.7072748494872092 0.2580838338745731 0.03464131663821786; 0.5469000332991828 0.3676841897174656 0.08541577698335168; 0.043089611741058076 0.020459821494324883 0.936450566764617], DriftDiffusionModel[DriftDiffusionModel(2.310049789802463, 7.426705555539929, 0.4919185808187338, 0.11710933059641579), DriftDiffusionModel(1.608015817225095, 1.042797104679359e-16, 0.5933711788805486, 0.13501818312550581), DriftDiffusionModel(0.9229246591063796, 7.019591731628739e-16, 0.5454964757518368, 0.2136101691962615)], [5.0 1.0 1.0; 1.0 5.0 1.0; 1.0 1.0 5.0], [2.0, 2.0, 2.0]), [-5466.84584393744, -841.4556464238941, -22.239716484968028, 502.56803265251057, 790.9239075695197, 964.9752666440503, 1080.1092742772755, 1165.3929371204833, 1229.6176043220244, 1282.3102828335034  …  1389.4373417199001, 1389.9730936383014, 1390.4366961060773, 1390.8387846376881, 1391.188106497203, 1391.491962931713, 1391.7566